In [1]:
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model

load_dotenv()

if not os.getenv("GOOGLE_API_KEY"):
    raise RuntimeError("GOOGLE_API_KEY is missing. Add it to .env or your shell environment.")

model = init_chat_model("google_genai:gemini-2.5-flash")


In [3]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage,SystemMessage
from langgraph.checkpoint.memory import MemorySaver

##message based summarization

agent = create_agent(
    model=model,
    checkpointer=MemorySaver(),
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger = ("messages",10),
            keep = ("messages",4)
        )
    ]
)


In [4]:
##runable thred

config = {
    "configurable":{"thread_id" : "1"}
}

In [5]:
questions = [
    "what is 2+1",
    "what is 2*2",
    "what is 2-34",
    "what is 2/4",
    "what is 2%5"
]

for q in questions:
    res = agent.invoke({
        "messages":[
            HumanMessage(content = q)]},config)
    print(f"message : {res}")
    print(f"message : {len(res['messages'])}")
    

message : {'messages': [HumanMessage(content='what is 2+1', additional_kwargs={}, response_metadata={}, id='32f01a31-58ba-4c1b-9525-f8e2775a6b77'), AIMessage(content='2 + 1 = 3', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f0256-66cb-7262-b960-24dc07200f0f-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 7, 'output_tokens': 35, 'total_tokens': 42, 'input_token_details': {'cache_read': 0}, 'output_token_details': {'reasoning': 28}})]}
message : 2
message : {'messages': [HumanMessage(content='what is 2+1', additional_kwargs={}, response_metadata={}, id='32f01a31-58ba-4c1b-9525-f8e2775a6b77'), AIMessage(content='2 + 1 = 3', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-2.5-flash', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f0256-66cb-7262-b960-24dc07200f0f-0', tool_cal

In [12]:
from langchain.agents import create_agent
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage,SystemMessage
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool

@tool
def get_Hotel(city : str):
    """sreach hotels -> return tokens"""
    return f"""
        Hotels in {city}:
        1.grand hotel - 5 star , $350/night, 4.8 rating
        2.Hilton Hotel - 4 star , $300/night, 4.5 rating
        3.Hotel California - 3 star , $250/night, 4.2 rating
        4.Hotel Sunshine - 2 star , $200/night, 4.0 rating
        """

agent = create_agent(
    model = model,
    tools = [get_Hotel],
    middleware=[
        SummarizationMiddleware(
            model=model,
            trigger =("tokens",500),
            keep = ("tokens",6)
        )
    ]
)
config = {
    "configurable":{"thread_id" : "test-1"}
}

def count_token(messages):
    total_char = sum(len(m.content)for m in messages)
    return total_char

In [ ]:
# cities = ["New York","London","Paris","Tokyo","Sydney"]

# for city in cities:
#     res = agent.invoke({
#         "messages":[
            
#             HumanMessage(content = f"find the hotels in {city}")
#         ]
#     })
    
#     # tokens = count_token(res["messages"])
#     # print(f"{city} ~{tokens} tokens,{len(res['messages'])}messages")
#     # print(f"{(res['messages'])}")

ChatGoogleGenerativeAIError: Error calling model 'gemini-2.5-flash' (RESOURCE_EXHAUSTED): 429 RESOURCE_EXHAUSTED. {'error': {'code': 429, 'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, head to: https://ai.google.dev/gemini-api/docs/rate-limits. To monitor your current usage, head to: https://ai.dev/rate-limit. \n* Quota exceeded for metric: generativelanguage.googleapis.com/generate_content_free_tier_requests, limit: 20, model: gemini-2.5-flash\nPlease retry in 24.938372968s.', 'status': 'RESOURCE_EXHAUSTED', 'details': [{'@type': 'type.googleapis.com/google.rpc.Help', 'links': [{'description': 'Learn more about Gemini API quotas', 'url': 'https://ai.google.dev/gemini-api/docs/rate-limits'}]}, {'@type': 'type.googleapis.com/google.rpc.QuotaFailure', 'violations': [{'quotaMetric': 'generativelanguage.googleapis.com/generate_content_free_tier_requests', 'quotaId': 'GenerateRequestsPerDayPerProjectPerModel-FreeT

### HUMANLOOP INTEGRATION


In [8]:
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langgraph.checkpoint.memory import MemorySaver
from langchain.messages import HumanMessage

def read_mail(email_id : str)->str:
    """read the email based on email_id"""
    return f"email content for id : {email_id}"

def send_mail(email_id : str , subject : str, body : str)->str:
    """send the email based on email_id"""
    return f"email sent for id : {email_id} , subject : {subject} , body : {body}"

In [9]:
agent = create_agent(
    model = model,
    tools= [read_mail,send_mail],
    checkpointer=MemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_mail" : {
                    "allowed_decisions" : ["approve", "edit", "reject"],
                },
                "read_mail" : False,    
            }
            
        )
    ]
)


In [10]:
config = {
    "configurable" : {
        "thread_id" : "test-approve"
    }
}

res = agent.invoke(
    {
        "messages" : [
            HumanMessage(content = "Send a mail to pramod4342@gmail.com with subject 'Greeting' and body 'Hey how are you?'")
        ]
    },
    config = config
)

interrupts = res.get("__interrupt__", ()) if isinstance(res, dict) else getattr(res, "interrupts", ())
if interrupts:
    print("Paused for approval:")
    print(interrupts[0].value)
else:
    print(res["messages"][-1].content)

Paused for approval:
{'action_requests': [{'name': 'send_mail', 'args': {'subject': 'Greeting', 'body': 'Hey how are you?', 'email_id': 'pramod4342@gmail.com'}, 'description': "Tool execution requires approval\n\nTool: send_mail\nArgs: {'subject': 'Greeting', 'body': 'Hey how are you?', 'email_id': 'pramod4342@gmail.com'}"}], 'review_configs': [{'action_name': 'send_mail', 'allowed_decisions': ['approve', 'edit', 'reject']}]}


In [11]:
from langgraph.types import Command

interrupts = res.get("__interrupt__", ()) if isinstance(res, dict) else getattr(res, "interrupts", ())
if interrupts:
    print("Paused! Approving...")
    res = agent.invoke(
        Command(
            resume= {
                "decisions":[
                    {"type" : "approve"}
                ]
            }
        ),
        config = config
    )
    print(f"result : {res['messages'][-1].content}")
else:
    print("No interrupt found; run the previous cell first, or check whether the model called send_mail.")

Paused! Approving...
result : I have sent the mail to pramod4342@gmail.com with subject 'Greeting' and body 'Hey how are you?'.


In [12]:
res

{'messages': [HumanMessage(content="Send a mail to pramod4342@gmail.com with subject 'Greeting' and body 'Hey how are you?'", additional_kwargs={}, response_metadata={}, id='eaab5cef-3d65-43ef-ad7a-b48b8a913ad9'),
  AIMessage(content='', additional_kwargs={'function_call': {'name': 'send_mail', 'arguments': '{"subject": "Greeting", "body": "Hey how are you?", "email_id": "pramod4342@gmail.com"}'}, '__gemini_function_call_thought_signatures__': {'efe03983-6b14-494e-af78-6dd54cb0e827': 'CtECAQw51se/EFlq3dZLqeG8a1kfMSJTiRtqVDFRIvtfbrQ+cp/HQ56WgJCdnG3AJ5K+qJ14VAZkNIqrV7qscJ1H5gHFI5llVNv3hrg8MvMUFaRXNbeWUFrT0HJsR8mptu1WF4j2IURjHnk7Alwl9hIpyanJd+N7+vee6nDpeu5tPgpKts5zFU5ckeaOiBHE2oENJnWNjkF9+kYvdfei5yi0ZG3DGNaY+IrvPr/UgdW5xgr6pXq4K5tr+6BtT8j0SVHWIQR1yu/MMmkDDBCxwvyHQFqAsT7469eL6UmJPH9X8z8bf/R9/qJqD1zM7RNAVa0Fesu+xDprXg01TjyKVIM4rLWzj8C3EqoY9lLpWviLUO0I5rZrf+8aGTy97tP8+5UBMzsKrnzrVCw9ik+cuMsbIy9j2+ClMWlBPndadFm+t2MefEU3Ckt3ggIc0usYvzUtNQ=='}}, response_metadata={'finish_reason': 'STOP', 'mode